In [26]:
# Mac settings
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

## Inference on one vid

In [ ]:
# Simple inference case
from transformers import VideoMAEVideoProcessor, VideoMAEForVideoClassification
import numpy as np
import torch
from huggingface_hub import hf_hub_download

video = list(np.random.randn(16, 3, 224, 224))

video_path = hf_hub_download(
    repo_id="nielsr/video-demo", filename="eating_spaghetti.mp4", repo_type="dataset"
)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
feature_extractor = VideoMAEVideoProcessor.from_pretrained("MCG-NJU/videomae-small-finetuned-ssv2")
model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-small-finetuned-ssv2")
model = model.to(device)

# inputs = feature_extractor(video, return_tensors="pt", device=device)
inputs = feature_extractor(video_path, do_sample_frames=True, num_frames=model.config.num_frames, return_tensors="pt", device=device)

with torch.no_grad():
  outputs = model(**inputs)
  logits = outputs.logits

predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])


Loading weights: 100%|██████████| 186/186 [00:00<00:00, 40470.02it/s]
/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/transformers/video_processing_utils.py:872: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/transformers/video_utils.py:534: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/Users/rodrigopaganini/miniconda3/envs/mva-t2c/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, whe

Predicted class: Piling [something] up


In [50]:
inputs['pixel_values'].shape

torch.Size([1, 16, 3, 224, 224])

## Text2Concept functionalities

In [63]:
import torchvision

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class VideoMAETToTWrapper(torch.nn.Module):
    def __init__(self, model, normalizer = None, mtype="videomae"):
        super().__init__()
        self.model = model
        self.normalizer = normalizer

    def forward_features(self, x):
        sequence_videomae_feats = self.model.videomae(**x).last_hidden_state

        if self.model.fc_norm is not None:
            videomae_feats = sequence_videomae_feats.mean(1)
            videomae_feats = self.model.fc_norm(videomae_feats)
        else:
            videomae_feats = sequence_videomae_feats[:, 0]

        return videomae_feats

    def get_normalizer(self, x):
        if self.normalizer is None:
            return x
        return self.normalizer(x)

    @property
    def has_normalizer(self):
        return self.normalizer is not None


device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
feature_extractor = VideoMAEVideoProcessor.from_pretrained("MCG-NJU/videomae-small-finetuned-ssv2")
model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-small-finetuned-ssv2")
model = model.to(device)

videomae_model = VideoMAETToTWrapper(model, normalizer=torchvision.transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))

Loading weights: 100%|██████████| 186/186 [00:00<00:00, 46478.44it/s]
